# Self-Attention — How Tokens Talk to Each Other

Notebook 005 ended with a comment buried in the FFN code:

> The FFN does NOT decide which tokens should interact with each other. That is the job of SELF-ATTENTION.

This notebook is that job.

The FFN (005) processes **every token independently** — token 3's vector goes through the same two linear layers whether token 1 said "cat" or "the stock market crashed." It has no way to look sideways at other tokens. Self-attention is the piece that lets a token **look at every other token** in the sequence and decide, for itself, which ones actually matter to it right now.


## Mental model: Query, Key, Value

The classic analogy is a **library lookup**, and it actually holds up:

- **Query (Q)** — "what am I looking for?" Every token asks this.
- **Key (K)** — "what do I have to offer?" Every token also advertises this.
- **Value (V)** — "here's my actual content, if you want it."

A token compares its **Query** against every other token's **Key** (including its own). A high match → that token's **Value** matters a lot to me. A low match → mostly ignore it.

```text
token's Query  ----compare against---->  every token's Key
                                                |
                                                v
                                     "how relevant is each token to me?"
                                                |
                                                v
                                weighted sum of every token's Value
                                                |
                                                v
                                    this token's new, context-aware vector
```

Q, K, V aren't three different things about a token conceptually — they're the **same token vector**, projected through three different learned linear layers. The projections are what let the model learn "what to look for" separately from "what to advertise" separately from "what to actually share."


In [1]:
import torch
import torch.nn as nn

torch.manual_seed(42)

# ---------------------------------------------------------
# Toy sequence: 3 tokens, embedding_dim = 4
# ---------------------------------------------------------
# Pretend this is "the cat sat" already embedded.
seq_len = 3
embedding_dim = 4

x = torch.randn(seq_len, embedding_dim)
print("x (our 3 token vectors):")
print(x)
print("x.shape:", x.shape)

x (our 3 token vectors):
tensor([[ 0.3367,  0.1288,  0.2345,  0.2303],
        [-1.1229, -0.1863,  2.2082, -0.6380],
        [ 0.4617,  0.2674,  0.5349,  0.8094]])
x.shape: torch.Size([3, 4])


In [2]:
# ---------------------------------------------------------
# Q, K, V = three separate learned linear projections
# ---------------------------------------------------------
# Same input x, three different nn.Linear layers.
# Each layer learns its own W and b.
W_q = nn.Linear(embedding_dim, embedding_dim, bias=False)
W_k = nn.Linear(embedding_dim, embedding_dim, bias=False)
W_v = nn.Linear(embedding_dim, embedding_dim, bias=False)

Q = W_q(x)   # "what is each token looking for?"
K = W_k(x)   # "what does each token offer?"
V = W_v(x)   # "each token's actual content"

print("Q.shape:", Q.shape)
print("K.shape:", K.shape)
print("V.shape:", V.shape)
# Q, K, V are all [seq_len, embedding_dim] — same shape as x.
# Nothing about the SHAPE changed yet. Only the VALUES did,
# because each went through a different linear layer.

Q.shape: torch.Size([3, 4])
K.shape: torch.Size([3, 4])
V.shape: torch.Size([3, 4])


## The formula

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Read it left to right, one piece at a time:

```text
Q, K          -->  QK^T            -->  raw similarity scores
                                          (how much does token i's query
                                           match token j's key?)

QK^T          -->  / sqrt(d_k)     -->  scaled scores
                                          (keeps the numbers from
                                           exploding — see below)

scaled scores -->  softmax(dim=-1) -->  attention weights
                                          (one row per token, weights
                                           sum to 1 across that row)

weights, V    -->  weights @ V     -->  output
                                          (each token's new vector =
                                           a weighted blend of every
                                           token's Value)
```

We'll build this one step at a time with real numbers instead of trusting the formula blind.


In [3]:
# ---------------------------------------------------------
# Step 1: QK^T -- raw similarity scores
# ---------------------------------------------------------
# Q is [seq_len, d_k], K.T is [d_k, seq_len]
# Q @ K.T -> [seq_len, seq_len]
#
# scores[i][j] = how much token i's Query matches token j's Key
raw_scores = Q @ K.T

print("raw_scores.shape:", raw_scores.shape)   # [3, 3] -- every token vs every token
print(raw_scores)

raw_scores.shape: torch.Size([3, 3])
tensor([[-0.0735, -0.2048, -0.2016],
        [ 0.3868,  1.6425,  0.9749],
        [-0.1760, -0.5046, -0.4773]], grad_fn=<MmBackward0>)


**Why `[seq_len, seq_len]`?** Every token needs a score against every other token — including itself. Row `i` is "how relevant is every token to token `i`." A `seq_len x seq_len` matrix is exactly that: one row per query token, one column per key token.


In [4]:
# ---------------------------------------------------------
# Step 2: scale by sqrt(d_k)
# ---------------------------------------------------------
# Why? Dot products grow with the number of dimensions being
# summed. Bigger d_k -> bigger raw scores -> softmax gets pushed
# into a near-one-hot corner (one weight ~1, the rest ~0) even
# when the actual similarities weren't that extreme. That kills
# gradients -- softmax is nearly flat everywhere except right at
# that corner.
#
# Dividing by sqrt(d_k) keeps the scores in a range where softmax
# still has real gradient signal.
d_k = K.shape[-1]
scaled_scores = raw_scores / (d_k ** 0.5)

print("d_k:", d_k)
print("scaled_scores:")
print(scaled_scores)

d_k: 4
scaled_scores:
tensor([[-0.0367, -0.1024, -0.1008],
        [ 0.1934,  0.8213,  0.4875],
        [-0.0880, -0.2523, -0.2387]], grad_fn=<DivBackward0>)


In [5]:
# ---------------------------------------------------------
# Step 3: softmax -- turn scores into attention weights
# ---------------------------------------------------------
attention_weights = torch.softmax(scaled_scores, dim=-1)

print("attention_weights:")
print(attention_weights)

# Sanity check: each ROW should sum to 1 -- it's a probability
# distribution over "which tokens should I pay attention to?"
print("\nrow sums (should all be 1.0):", attention_weights.sum(dim=-1))

attention_weights:
tensor([[0.3479, 0.3258, 0.3263],
        [0.2372, 0.4445, 0.3183],
        [0.3692, 0.3133, 0.3176]], grad_fn=<SoftmaxBackward0>)

row sums (should all be 1.0): tensor([1., 1., 1.], grad_fn=<SumBackward1>)


In [6]:
# ---------------------------------------------------------
# Step 4: weighted sum of V
# ---------------------------------------------------------
# attention_weights: [seq_len, seq_len]
# V:                 [seq_len, embedding_dim]
# weights @ V ->      [seq_len, embedding_dim]
#
# Row i of the output = a BLEND of every token's Value, weighted
# by how much token i attended to it.
output = attention_weights @ V

print("output.shape:", output.shape)   # same shape as x -- [3, 4]
print(output)

output.shape: torch.Size([3, 4])
tensor([[0.3172, 0.0995, 0.3417, 0.4694],
        [0.4189, 0.1614, 0.4470, 0.6191],
        [0.3071, 0.0939, 0.3310, 0.4521]], grad_fn=<MmBackward0>)


**Notice the shape.** `output.shape == x.shape`. Self-attention takes `[seq_len, embedding_dim]` in and hands back `[seq_len, embedding_dim]` out — same shape as the FFN in 005. That's exactly why the two can stack: attention's output can feed straight into an FFN, and (with a residual connection) back into another attention layer. Shape-preservation is what makes "stack N of these blocks" even possible.


## Causal masking — why an LLM can't just use this as-is

Everything above lets **every token see every other token**, including tokens that come *after* it. That's fine for something like a sentiment classifier that gets the whole sentence at once. It is **not** fine for an LLM predicting the next token — if token 3 is allowed to peek at token 4's Key/Value while training to *predict* token 4, it's cheating: it's using the answer to predict the answer.

**Causal masking** fixes this: before the softmax, force every "future" position's score to `-inf`. `softmax(-inf) = 0`, so those tokens get exactly zero attention weight — token `i` can only attend to tokens `0..i`, never `i+1..end`.

```text
allowed (lower triangle):        blocked (upper triangle):

token 0: [0]                     token 0: [ -, -, -]   (can't see 1, 2)
token 1: [0, 1]                  token 1: [ 0, -, -]   (can't see 2)
token 2: [0, 1, 2]                token 2: [ 0, 1, -]
```


In [7]:
# ---------------------------------------------------------
# Building the causal mask
# ---------------------------------------------------------
# torch.tril = lower triangle (including diagonal) kept, rest zeroed.
# We use it to build a boolean mask of "allowed" positions.
mask = torch.tril(torch.ones(seq_len, seq_len)).bool()
print("mask (True = allowed to attend):")
print(mask)

mask (True = allowed to attend):
tensor([[ True, False, False],
        [ True,  True, False],
        [ True,  True,  True]])


In [8]:
# ---------------------------------------------------------
# Apply the mask BEFORE softmax, not after
# ---------------------------------------------------------
# masked_fill replaces every position where mask is False with
# -inf. Softmax of -inf is 0, so those positions contribute
# nothing to the weighted sum -- but they still exist as zeros,
# not as "removed" positions, so the tensor shape never changes.
masked_scores = scaled_scores.masked_fill(mask == False, float("-inf"))
print("masked_scores:")
print(masked_scores)

causal_weights = torch.softmax(masked_scores, dim=-1)
print("\ncausal_weights:")
print(causal_weights)
print("\nrow sums (still 1.0 -- softmax over fewer, unmasked positions):", causal_weights.sum(dim=-1))

masked_scores:
tensor([[-0.0367,    -inf,    -inf],
        [ 0.1934,  0.8213,    -inf],
        [-0.0880, -0.2523, -0.2387]], grad_fn=<MaskedFillBackward0>)

causal_weights:
tensor([[1.0000, 0.0000, 0.0000],
        [0.3480, 0.6520, 0.0000],
        [0.3692, 0.3133, 0.3176]], grad_fn=<SoftmaxBackward0>)

row sums (still 1.0 -- softmax over fewer, unmasked positions): tensor([1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)


Look at row 0 of `causal_weights`: it's `[1.0, 0.0, 0.0]`. Token 0 has no past to look at, so it can only attend to itself — mechanically forced, not learned. Row 2 is the only row with weight spread across all three positions, because token 2 is the only one that's allowed to see everything.

This one mask is the entire mechanical difference between an **encoder** (BERT-style, no mask, sees the whole sequence) and a **decoder** (GPT-style, causal mask, only ever sees the past). Same attention math either way.


## Packaging it: a `SelfAttention` module

Same pattern as 004's `LinearRegressionModel` — wrap the steps above in an `nn.Module` so it's a reusable brick instead of loose cells.


In [9]:
class SelfAttention(nn.Module):

    def __init__(self, embedding_dim, causal=True):
        super().__init__()

        self.embedding_dim = embedding_dim
        self.causal = causal

        # Same three projections as above, now owned by the module
        # so PyTorch tracks their weights as learnable parameters.
        self.W_q = nn.Linear(embedding_dim, embedding_dim, bias=False)
        self.W_k = nn.Linear(embedding_dim, embedding_dim, bias=False)
        self.W_v = nn.Linear(embedding_dim, embedding_dim, bias=False)

    def forward(self, x):
        # x: [seq_len, embedding_dim]
        seq_len = x.shape[0]

        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        d_k = K.shape[-1]
        scores = (Q @ K.T) / (d_k ** 0.5)

        if self.causal:
            mask = torch.tril(torch.ones(seq_len, seq_len)).bool()
            scores = scores.masked_fill(mask == False, float("-inf"))

        weights = torch.softmax(scores, dim=-1)
        output = weights @ V

        return output

In [10]:
torch.manual_seed(0)

attn = SelfAttention(embedding_dim=4, causal=True)
out = attn(x)

print("input  shape:", x.shape)
print("output shape:", out.shape)
print(out)

input  shape: torch.Size([3, 4])
output shape: torch.Size([3, 4])
tensor([[-0.3285,  0.0652, -0.1119, -0.0134],
        [-0.1867,  0.3463,  0.2201,  0.5140],
        [-0.3949,  0.3172, -0.0025,  0.3742]], grad_fn=<MmBackward0>)


## Multi-Head Attention — why one head isn't enough

A single attention head learns **one** notion of "relevance." But language has many simultaneous relationships in the same sentence — "The cat sat because it was tired" needs *"it" -> "cat"* (coreference) tracked at the same time as *"sat" -> "because...tired"* (causality). One softmax distribution per token can't cleanly represent multiple, unrelated relationships at once — it would have to smear them into one averaged blend.

**Multi-head attention** runs several smaller attention operations in parallel, each with its own learned Q/K/V projections, then concatenates the results:

```text
embedding_dim=8, num_heads=2
                                         head 1 (dims 0-3): attends to ???
input [seq_len, 8]  --split into 2-->
                                         head 2 (dims 4-7): attends to ???
                                                    |
                    concat back to [seq_len, 8]  <--+
                                    |
                        final linear projection (mix the heads together)
                                    |
                                output [seq_len, 8]
```

Each head gets a **smaller** slice of the embedding (`embedding_dim / num_heads`), so the total compute stays roughly the same as one big head — you're trading one wide attention op for several narrow ones running side by side.


In [11]:
class MultiHeadAttention(nn.Module):

    def __init__(self, embedding_dim, num_heads, causal=True):
        super().__init__()

        assert embedding_dim % num_heads == 0, "embedding_dim must divide evenly across heads"

        self.num_heads = num_heads
        self.head_dim = embedding_dim // num_heads
        self.causal = causal

        # One big projection each for Q, K, V -- we split into heads
        # AFTER projecting, rather than making a separate nn.Linear
        # per head. Mathematically equivalent, one matmul instead of
        # num_heads small ones.
        self.W_q = nn.Linear(embedding_dim, embedding_dim, bias=False)
        self.W_k = nn.Linear(embedding_dim, embedding_dim, bias=False)
        self.W_v = nn.Linear(embedding_dim, embedding_dim, bias=False)

        # Mixes the concatenated heads back together at the end.
        self.out_proj = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, x):
        seq_len, embedding_dim = x.shape

        Q = self.W_q(x)   # [seq_len, embedding_dim]
        K = self.W_k(x)
        V = self.W_v(x)

        # Split embedding_dim into (num_heads, head_dim), then move
        # num_heads in front of seq_len so each head's [seq_len, head_dim]
        # slab can be attended to independently.
        # [seq_len, embedding_dim] -> [seq_len, num_heads, head_dim] -> [num_heads, seq_len, head_dim]
        Q = Q.view(seq_len, self.num_heads, self.head_dim).transpose(0, 1)
        K = K.view(seq_len, self.num_heads, self.head_dim).transpose(0, 1)
        V = V.view(seq_len, self.num_heads, self.head_dim).transpose(0, 1)

        # Same scaled dot-product attention as SelfAttention above,
        # just batched over the num_heads dimension now.
        scores = (Q @ K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        # scores: [num_heads, seq_len, seq_len]

        if self.causal:
            mask = torch.tril(torch.ones(seq_len, seq_len)).bool()
            scores = scores.masked_fill(mask == False, float("-inf"))

        weights = torch.softmax(scores, dim=-1)
        head_outputs = weights @ V   # [num_heads, seq_len, head_dim]

        # Put the heads back side by side and merge them back into
        # one embedding_dim-wide vector per token.
        # [num_heads, seq_len, head_dim] -> [seq_len, num_heads, head_dim] -> [seq_len, embedding_dim]
        concat = head_outputs.transpose(0, 1).reshape(seq_len, embedding_dim)

        return self.out_proj(concat)

In [12]:
torch.manual_seed(0)

mha = MultiHeadAttention(embedding_dim=8, num_heads=2, causal=True)
x8 = torch.randn(seq_len, 8)

out = mha(x8)

print("input  shape:", x8.shape)
print("output shape:", out.shape)   # same shape in and out, same rule as SelfAttention
print(out)

input  shape: torch.Size([3, 8])
output shape: torch.Size([3, 8])
tensor([[ 0.2285,  0.5819, -0.2551, -0.2788,  0.1080,  0.2809,  0.3251,  0.2878],
        [-0.1045,  0.2218, -0.2381, -0.3404, -0.0415,  0.1945,  0.2942,  0.3317],
        [-0.2232,  0.1314, -0.2214, -0.4507, -0.1367,  0.1728,  0.3067,  0.3262]],
       grad_fn=<AddmmBackward0>)


## Where this fits

Every brick needed for one Transformer block now exists across these two notebooks:

```text
input embeddings
      |
      +--> Multi-Head Attention (this notebook) -- tokens look at each other
      |
      +--> Feed-Forward Network (005)            -- each token processed independently
      |
      v
  richer token representations
```

Still missing before this is a real Transformer block: **residual connections** (add the input back to the output, so gradients have a direct path through every layer) and **LayerNorm** (004 already covers what it does) wrapped around both attention and the FFN, plus **positional encoding** — attention has no built-in notion of token order (swap two rows of `x` above and the math doesn't know the difference), which a sequence obviously needs. Those, plus stacking several blocks and adding tokenization, are the next steps toward an actual mini-GPT.
